# Joints and frames

This module reads the robot's joints, moves one of them, and works out
**where the hand is** from the joint angles (forward kinematics), then checks
that against a physics engine's answer.

You need: `g1lab` (see the [README](../README.md)), `numpy`, `matplotlib`, and `mujoco`
for the last part and for the mock. Read [the guide](../docs/guide.md)
first, at least "A first program".

**Run it on the mock first** (`HOST = "mock"`, on your laptop). The same
notebook runs on the simulator and on the robot: only `HOST` changes.

In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME) if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. Read the joints

The robot has 29 body joints. You can command 14 of them: 7 per arm, in this
order. Reading needs no session and no turn.

In [ ]:
print("arm joints:", g1lab.ARM)
for group in ("left_arm", "right_arm"):
    lo, hi = info.limits[group].T              # (7, 2): low, high
    print(f"\n{group}")
    for name, l, h, q0 in zip(info.groups[group], lo, hi, info.home[group]):
        print(f"  {name:15s} {math.degrees(l):7.1f} .. {math.degrees(h):6.1f} deg   home {math.degrees(q0):6.1f}")

js = r.state.joints()
print("\nright arm now (deg):", np.round(np.degrees(js.arm("right")), 1))
print("tilt (roll, pitch, yaw, deg):", np.round(np.degrees(r.state.imu().rpy), 1))

Start every pose from `info.home` and change only the joints you care about.
An all-zero pose is refused: it puts the two hands almost together.

## 2. Move one joint

We swing the right arm forward by 0.8 rad at the shoulder (one joint,
`shoulder_pitch`) and back, and record every reading on the way. Negative
pitch is forward. `target` is what your program sends, `q` what the robot
measures.

In [ ]:
class JointLog:
    """Records every joint reading (50 a second) while it is running:
    t (this computer's clock, when the reading arrived), measured q (29)
    and target q (29). Given c (your r.control()), target is what your
    program was sending for the arms at that moment; without it, what the
    robot reports it was asked for (the simulator reports nothing: NaN)."""
    def __init__(self, robot, c=None):
        self.rows, self._c = [], c
        self._stop = robot.on("joints", self._add)
    def _add(self, js):
        tgt = js.target if js.target is not None else np.full(len(js.q), np.nan)
        if self._c is not None:
            tgt = np.full(len(js.q), np.nan)
            for side, first in (("left", "L_sh_pitch"), ("right", "R_sh_pitch")):
                i = g1lab.BODY.index(first)
                tgt[i:i + 7] = self._c.arms.target(side)
        self.rows.append((time.time(), js.q.copy(), np.asarray(tgt, float).copy()))
    def stop(self, t0=None):
        """Stop recording: (t, q, target), t in s from t0 (default: the first reading)."""
        self._stop()
        t = np.array([row[0] for row in self.rows])
        return t - (t[0] if t0 is None else t0), np.array([row[1] for row in self.rows]), np.array([row[2] for row in self.rows])

In [ ]:
PITCH = g1lab.BODY.index("R_sh_pitch")
home = info.home["right_arm"].copy()
forward = home.copy()
forward[0] -= 0.8                                   # shoulder_pitch: forward

with r.control() as c:                              # waits for your turn, then eases in
    log = JointLog(r, c)
    c.arms.move_to(right=forward, duration=2.0)
    time.sleep(1.0)
    c.arms.move_to(right=home, duration=2.0)
    time.sleep(1.0)
t, q, target = log.stop()
print(len(t), "readings over", round(t[-1], 1), "s")

plt.figure(figsize=(8, 3))
plt.plot(t, np.degrees(target[:, PITCH]), label="target")
plt.plot(t, np.degrees(q[:, PITCH]), label="measured")
plt.xlabel("time (s)"); plt.ylabel("right shoulder pitch (deg)"); plt.legend(); plt.grid(alpha=.3)
plt.show()

**How far behind is the arm?** The measured angle trails the target. The next
cell tries shifts of 0 to 0.5 s and keeps the one with the smallest error.

In [ ]:
ok = ~np.isnan(target[:, PITCH])
best = None
for shift in np.arange(0.0, 0.5, 0.01):
    shifted = np.interp(t[ok] - shift, t[ok], target[ok, PITCH])
    err = np.sqrt(np.mean((shifted - q[ok, PITCH]) ** 2))
    if best is None or err < best[1]:
        best = (shift, err)
print(f"measured lags the target by about {best[0]:.2f} s (rms error then {math.degrees(best[1]):.2f} deg)")

## 3. Forward kinematics

The robot's description file (URDF) lists every joint: which link it joins
to which, where it sits on its parent (`origin`: a shift `xyz` and a turn
`rpy`) and the axis it turns about. Chaining them from the pelvis out gives
the pose of every link.

For one revolute joint at angle *q*:
T<sub>child</sub> = T<sub>parent</sub> · T<sub>origin</sub> · Rot(axis, *q*).

The URDF comes with `g1lab`.

In [ ]:
import xml.etree.ElementTree as ET

URDF = os.path.join(os.path.dirname(g1lab.__file__), "description", "g1_29dof_dx3_ros.urdf")
urdf = ET.parse(URDF).getroot()
JOINT_OF_CHILD = {j.find("child").get("link"): j for j in urdf.findall("joint")}

def chain(link):
    """The joints from the pelvis out to `link`, in order."""
    out = []
    while link in JOINT_OF_CHILD:
        j = JOINT_OF_CHILD[link]
        out.append(j)
        link = j.find("parent").get("link")
    return out[::-1]

for j in chain("right_hand_palm_link"):
    o = j.find("origin")
    print(f"{j.get('name'):28s} {j.get('type'):9s} xyz {o.get('xyz'):32s} rpy {o.get('rpy')}")

`fk(angles, link)` below gives the 4×4 pose of `link` in the pelvis frame, for
a dict `angles` of URDF joint name → angle (missing joints are 0). It needs a
rotation from roll-pitch-yaw
(R = R<sub>z</sub>(yaw) R<sub>y</sub>(pitch) R<sub>x</sub>(roll)) and a
rotation about an axis (Rodrigues' formula).

In [ ]:
def rot_axis(axis, q):
    k = np.asarray(axis, float) / np.linalg.norm(axis)
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + math.sin(q) * K + (1 - math.cos(q)) * K @ K

def rot_rpy(roll, pitch, yaw):
    return rot_axis([0, 0, 1], yaw) @ rot_axis([0, 1, 0], pitch) @ rot_axis([1, 0, 0], roll)

def numbers(s):
    return [float(v) for v in s.split()]

def fk(angles, link):
    T = np.eye(4)
    for j in chain(link):
        o = j.find("origin")
        step = np.eye(4)
        step[:3, :3] = rot_rpy(*numbers(o.get("rpy", "0 0 0")))
        step[:3, 3] = numbers(o.get("xyz", "0 0 0"))
        T = T @ step
        if j.get("type") == "revolute":
            turn = np.eye(4)
            turn[:3, :3] = rot_axis(numbers(j.find("axis").get("xyz")), angles.get(j.get("name"), 0.0))
            T = T @ turn
    return T

In [ ]:
def urdf_angles(js):
    """A joint reading -> {URDF joint name: angle} for the waist and both arms."""
    a = {f"waist_{n}_joint": js[f"waist_{n}"] for n in ("yaw", "roll", "pitch")}
    for side in ("left", "right"):
        for name, q in zip(g1lab.ARM, js.arm(side)):
            a[f"{side}_{name}_joint"] = float(q)
    return a

T = fk(urdf_angles(r.state.joints()), "right_hand_palm_link")
print("right palm in the pelvis frame (m):", np.round(T[:3, 3], 3))

The pelvis frame has x forward, y to the robot's left and z up. The right palm is on the robot's right, so its y is negative. Swinging the arm forward raises x and z and barely changes y: the plot at the end of part 4 shows it.

## 4. Check against MuJoCo

MuJoCo has its own model of the robot (it comes with `g1lab` too). We put
its joints at the measured angles and ask where the wrist is. MuJoCo's
model has no palm link, so we compare the last wrist link,
`right_wrist_yaw_link`. Your FK and MuJoCo's should agree to well under a
millimetre.

In [ ]:
import mujoco

MJCF = os.path.join(os.path.dirname(g1lab.__file__), "model", "g1_29dof_with_hand.xml")
model = mujoco.MjModel.from_xml_path(MJCF)
data = mujoco.MjData(model)

def mujoco_pose(angles, body):
    """`body`'s 4x4 pose in the pelvis frame, from MuJoCo."""
    data.qpos[:] = 0
    data.qpos[3] = 1.0                       # the floating base: upright (quaternion w)
    for name, q in angles.items():
        data.qpos[model.joint(name).qposadr[0]] = q
    mujoco.mj_kinematics(model, data)
    def pose(b):
        T = np.eye(4)
        T[:3, :3] = data.body(b).xmat.reshape(3, 3)
        T[:3, 3] = data.body(b).xpos
        return T
    return np.linalg.inv(pose("pelvis")) @ pose(body)

rows = []
with r.control() as c:
    for pitch, elbow in ((0.0, 0.0), (-0.4, 0.4), (-0.8, 0.6)):
        pose = home.copy(); pose[0] += pitch; pose[3] += elbow   # forward, then bend
        c.arms.move_to(right=pose, duration=1.5)
        time.sleep(0.5)
        angles = urdf_angles(r.state.joints())
        mine = fk(angles, "right_wrist_yaw_link")[:3, 3]
        theirs = mujoco_pose(angles, "right_wrist_yaw_link")[:3, 3]
        rows.append((pitch, elbow, mine, theirs))

for pitch, elbow, mine, theirs in rows:
    print(f"pitch {pitch:+.1f} elbow {elbow:+.1f}: mine {np.round(mine, 4)}  mujoco {np.round(theirs, 4)}"
          f"  difference {1000 * np.linalg.norm(mine - theirs):.3f} mm")
assert max(np.linalg.norm(m - t) for _, _, m, t in rows) < 1e-3, "FK and MuJoCo disagree"

**The palm's path.** The readings from part 2 go through `fk()` as well, so
you can plot where the right palm went (x and z) while the shoulder swung.

In [ ]:
names = g1lab.BODY
path = []
for row in q:
    a = {f"waist_{n}_joint": row[names.index(f"waist_{n}")] for n in ("yaw", "roll", "pitch")}
    for name, v in zip(g1lab.ARM, row[22:29]):
        a[f"right_{name}_joint"] = v
    path.append(fk(a, "right_hand_palm_link")[:3, 3])
path = np.array(path)
plt.figure(figsize=(4, 4))
plt.plot(path[:, 0], path[:, 2])
plt.xlabel("x forward (m)"); plt.ylabel("z up (m)"); plt.axis("equal"); plt.grid(alpha=.3)
plt.title("right palm while the shoulder swings"); plt.show()

## Letting go

Leaving each `with r.control()` block already let go of the robot: the arms
went home and the next person in line got it.

**Try this.** Change `forward` in part 2 to swing a different joint, such as
the elbow: `forward[3] -= 0.8` instead of `forward[0]`, and
`PITCH = g1lab.BODY.index("R_elbow")`. Run part 2 again and the two cells that use
it, and see how the delay and the palm's path change.